In [1]:
import xarray as xr

In [ ]:
import os
import numpy as np
import xarray as xr
from datetime import datetime, timedelta

# Ruta de la carpeta donde están los archivos mensuales
base_path = "2025/"  # Actualiza esta ruta

# Ruta de la carpeta donde se guardarán los archivos diarios
output_base_dir = "2025/descompuestos/"  # Actualiza esta ruta

# Asegúrate de que el directorio de salida exista
if not os.path.exists(output_base_dir):
    os.makedirs(output_base_dir)

# Nombres de las variables
variables = ['e', 'ml_q', 'ml_u', 'ml_v', 'sp', 'tcw', 'tp']

# Función para descomponer los archivos
def descomponer_archivos():
    for mes in range(1, 13):  # Para cada mes del 01 al 12
        mes_str = f"{mes:02d}"  # Asegurarse de que el mes tenga dos dígitos
        mes_path = os.path.join(base_path, mes_str)

        # Verificar si la carpeta mensual existe
        if not os.path.exists(mes_path):
            print(f"Carpeta del mes {mes_str} no encontrada.")
            continue
        
        # Crear la carpeta de salida para el mes
        output_dir = os.path.join(output_base_dir, mes_str)
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)

        for var in variables:  # Para cada variable
            archivo_mensual = os.path.join(mes_path, f"ERA5_2025-{mes_str}_{var}.nc")
            
            # Verificar si el archivo existe
            if not os.path.exists(archivo_mensual):
                print(f"Archivo {archivo_mensual} no encontrado.")
                continue

            # Abrir el archivo .nc
            with xr.open_dataset(archivo_mensual) as ds:
                # Eliminar las coordenadas 'expver' y 'number' si están presentes
                if 'expver' in ds.coords:
                    ds = ds.drop_vars(['expver'])
                if 'number' in ds.coords:
                    ds = ds.drop_vars(['number'])

                # Convertir las coordenadas 'latitude' y 'longitude' a float32
                if 'latitude' in ds.coords:
                    ds['latitude'] = ds['latitude'].astype(np.float32)
                if 'longitude' in ds.coords:
                    ds['longitude'] = ds['longitude'].astype(np.float32)

                # Convertir las variables a float64
                for var_name in ds.data_vars:
                    ds[var_name] = ds[var_name].astype(np.float64)

                # Cambiar 'ps' a 'sp' si la variable es 'ps'
                if 'ps' in ds.data_vars:
                    ds = ds.rename({'ps': 'sp'})

                # Cambiar 'valid_time' a 'time'
                if 'valid_time' in ds.coords:
                    ds = ds.rename({'valid_time': 'time'})

                # Reordenar las dimensiones a 'longitude', 'latitude', 'time' y 'model_level' (si es relevante)
                if 'model_level' in ds.dims:
                    ds = ds.transpose('longitude', 'latitude', 'time', 'model_level')
                    ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) -180))
                else:
                    ds = ds.transpose('time', 'latitude', 'longitude')

                # Iterar por cada día en el mes
                for dia in np.unique(ds['time'].dt.date.values):
                    # Filtrar los datos para el día actual
                    dia_str = str(dia)
                    ds_dia = ds.sel(time=ds['time'].dt.date == dia)

                    # Crear el archivo de salida con las dimensiones correctas
                    output_filename = f"ERA5_{dia_str}_{var}.nc"
                    output_path = os.path.join(output_dir, output_filename)

                    # Guardar el archivo con los datos del día
                    ds_dia.to_netcdf(output_path)
                    print(f"Archivo guardado: {output_path}")

# Llamar a la función para realizar la descomposición
descomponer_archivos()

Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-01_e.nc
Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-02_e.nc
Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-03_e.nc
Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-04_e.nc
Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-05_e.nc
Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-06_e.nc
Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-07_e.nc
Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-08_e.nc
Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-09_e.nc
Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-10_e.nc
Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-11_e.nc
Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-12_e.nc
Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-13_e.nc
Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-14_e.nc
Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-15_e.nc
Archivo guardado: 2025/descompuestos/01/ERA5_2025-01-16_e.nc
Archivo guardado: 2025/d